In [ ]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TA indicators
import ta
from ta.trend import PSARIndicator

# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:



#  "NVDA"

symbols = [
    "NVDA", "GOOGL", "AAPL", 
    "GOOG", "MSFT", "AVGO", 
    "TSLA", "META"
]


dfs = {}

for s in symbols:
    filepath = f"NASDAQ_DATA/{s}_daily.csv"
    
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue
    
    df = pd.read_csv(filepath)
    dfs[s] = df
    print(f"\n── {s} ──────────────────────────")
    print(f"  Shape : {df.shape}")
    print(f"  Head  :\n{df.head()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"  Nulls :\n{df.isnull().sum()}")

print(f"\n── Loaded {len(dfs)}/{len(symbols)} tickers ──")


── NVDA ──────────────────────────
  Shape : (6683, 62)
  Head  :
              datetime ticker  open_price  high_price  low_price  close_price  \
0  1999-11-04 22:30:00   NVDA    0.057813    0.062370   0.057813     0.060808   
1  1999-11-05 22:30:00   NVDA    0.062500    0.063021   0.055990     0.058854   
2  1999-11-08 22:30:00   NVDA    0.057031    0.062240   0.055209     0.060547   
3  1999-11-09 22:30:00   NVDA    0.060417    0.060677   0.057292     0.059636   
4  1999-11-10 22:30:00   NVDA    0.059831    0.059896   0.057943     0.059115   

         volume    EMA_10    EMA_20  EMA_ratio  ...  volume_lag_3  \
0  1.260330e+09  0.050692  0.048170   1.262353  ...  7.825401e+08   
1  6.163649e+08  0.052176  0.049188   1.196531  ...  8.374998e+08   
2  4.700136e+08  0.053698  0.050269   1.204453  ...  2.011670e+09   
3  2.723506e+08  0.054778  0.051161   1.165637  ...  1.260330e+09   
4  1.440472e+08  0.055566  0.051919   1.138600  ...  6.163649e+08   

   volume_lag_4  volume_lag_5  

In [3]:
for symbol in symbols:
    df = pd.read_csv(f'NASDAQ_DATA/{symbol}_daily.csv', parse_dates=["datetime"])

    df = df.dropna().reset_index(drop=True)

    # Outlier removal — price only
    mean, std = df["price_pct_change"].mean(), df["price_pct_change"].std()
    df = df[df["price_pct_change"].between(mean - 3 * std, mean + 3 * std)]
    df = df.reset_index(drop=True)

    # Split
    n        = len(df)
    train_df = df.iloc[:int(n * 0.80)]
    val_df   = df.iloc[int(n * 0.80):int(n * 0.90)]
    test_df  = df.iloc[int(n * 0.90):]

    # Scale for diagnostics
    target_scaler = StandardScaler()
    train_y = target_scaler.fit_transform(train_df[["price_pct_change"]])
    val_y   = target_scaler.transform(val_df[["price_pct_change"]])

    print(f"\n{symbol}")
    print(f"  Raw train UP%      : {(train_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Raw val UP%        : {(val_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Scaled train mean  : {train_y[:, 0].mean():.4f}")
    print(f"  Scaled train std   : {train_y[:, 0].std():.4f}")
    print(f"  Scaled val mean    : {val_y[:, 0].mean():.4f}")
    print(f"  Scaled val std     : {val_y[:, 0].std():.4f}")
    print(f"  Train date range   : {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Val date range     : {val_df['datetime'].min().date()} → {val_df['datetime'].max().date()}")


NVDA
  Raw train UP%      : 51.18%
  Raw val UP%        : 52.27%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : 0.0084
  Scaled val std     : 1.0240
  Train date range   : 1999-11-04 → 2021-02-17
  Val date range     : 2021-02-18 → 2023-10-06

GOOGL
  Raw train UP%      : 52.26%
  Raw val UP%        : 51.73%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : 0.0144
  Scaled val std     : 1.2005
  Train date range   : 2005-06-03 → 2022-03-11
  Val date range     : 2022-03-14 → 2024-04-23

AAPL
  Raw train UP%      : 52.16%
  Raw val UP%        : 51.41%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : -0.0386
  Scaled val std     : 0.8016
  Train date range   : 1999-03-12 → 2021-01-06
  Val date range     : 2021-01-07 → 2023-09-15

GOOG
  Raw train UP%      : 53.58%
  Raw val UP%        : 56.38%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : -0.050

In [4]:
import pickle
import os

SEQ_LEN_DEFAULT = 30
prepared = {}
scalers  = {}

FEATURE_COLS = [
    "open_price", "high_price", "low_price", "close_price", "volume",
    # Trend
    "EMA_10", "EMA_20", "EMA_ratio", "EMA_50_ratio", "EMA_200_ratio", "MACD_hist",
    # Momentum
    "RSI_14", "ROC_5", "ROC_10", "ROC_20",
    # Directional movement
    "ADX_14", "DI_plus", "DI_minus",
    # Volatility
    "ATR_14", "ATR_ratio", "BB_pct", "BB_width", "realized_vol",
    # Volume
    "OBV", "OBV_momentum", "volume_ratio", "volume_surge", "MFI_14", "CMF_20",
    # Price action
    "overnight_gap", "intraday_direction", "upper_shadow", "lower_shadow",
    # Range / calendar
    "price_percentile_52w", "dow_sin", "dow_cos",
    # Market context
    "QQQ_return_lag1", "SPY_return_lag1",
    # Lags
    "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_4", "close_lag_5",
    "returns_lag_1", "returns_lag_2", "returns_lag_3", "returns_lag_4", "returns_lag_5",
    "volume_lag_1", "volume_lag_2", "volume_lag_3", "volume_lag_4", "volume_lag_5",
    # Regime
    "above_ema10", "above_ema20", "roc5_pos", "roc20_pos", "up_streak",
]

TARGET_COLS = ["price_pct_change"]   # single target — price only


def _make_seqs(X, y, seq_len):
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        ws.append(1.0)
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


for symbol in symbols:
    filepath = f"NASDAQ_DATA/{symbol}_daily.csv"
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue

    df = pd.read_csv(filepath, parse_dates=["datetime"])

    df = df.dropna().reset_index(drop=True)

    # ── Clip outliers — price target only ─────────────────────────────────────
    mean, std = df["price_pct_change"].mean(), df["price_pct_change"].std()
    df = df[df["price_pct_change"].between(mean - 3 * std, mean + 3 * std)]
    df = df.reset_index(drop=True)

    # ── Chronological 80 / 10 / 10 split ─────────────────────────────────────
    n        = len(df)
    train_df = df.iloc[:int(n * 0.80)]
    val_df   = df.iloc[int(n * 0.80):int(n * 0.90)]
    test_df  = df.iloc[int(n * 0.90):]

    train_X_raw = train_df[FEATURE_COLS].values.astype(np.float32)
    val_X_raw   = val_df[FEATURE_COLS].values.astype(np.float32)
    test_X_raw  = test_df[FEATURE_COLS].values.astype(np.float32)

    train_y_raw = train_df[TARGET_COLS].values.astype(np.float32)  # (N, 1)
    val_y_raw   = val_df[TARGET_COLS].values.astype(np.float32)
    test_y_raw  = test_df[TARGET_COLS].values.astype(np.float32)

    # ── Fit scalers on train only ─────────────────────────────────────────────
    feature_scaler = StandardScaler()
    target_scaler  = StandardScaler()

    train_X_scaled = feature_scaler.fit_transform(train_X_raw)
    val_X_scaled   = feature_scaler.transform(val_X_raw)
    test_X_scaled  = feature_scaler.transform(test_X_raw)

    train_y_scaled = target_scaler.fit_transform(train_y_raw)
    val_y_scaled   = target_scaler.transform(val_y_raw)
    test_y_scaled  = target_scaler.transform(test_y_raw)

    # ── Save scalers ──────────────────────────────────────────────────────────
    os.makedirs(f"models/{symbol}", exist_ok=True)
    with open(f"models/{symbol}/{symbol}_feature_scaler.pkl", "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "wb") as f:
        pickle.dump(target_scaler, f)

    # ── Pre-build sequences at default SEQ_LEN ────────────────────────────────
    X_train, y_train, _ = _make_seqs(train_X_scaled, train_y_scaled, SEQ_LEN_DEFAULT)
    X_val,   y_val,   _ = _make_seqs(val_X_scaled,   val_y_scaled,   SEQ_LEN_DEFAULT)
    X_test,  y_test,  _ = _make_seqs(test_X_scaled,  test_y_scaled,  SEQ_LEN_DEFAULT)

    prepared[symbol] = {
        "train_X_scaled": train_X_scaled,
        "train_y_scaled": train_y_scaled,
        "val_X_scaled":   val_X_scaled,
        "val_y_scaled":   val_y_scaled,
        "test_X_scaled":  test_X_scaled,
        "test_y_scaled":  test_y_scaled,
        "X_train": X_train,  "y_train": y_train,
        "X_val":   X_val,    "y_val":   y_val,
        "X_test":  X_test,   "y_test":  y_test,
    }
    scalers[symbol] = {
        "feature_scaler": feature_scaler,
        "target_scaler":  target_scaler,
    }

    print(f"\n── {symbol} {'─' * 60}")
    print(f"  Rows  → train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")
    print(f"  Seqs  → train:{X_train.shape[0]}  val:{X_val.shape[0]}  test:{X_test.shape[0]}")
    print(f"  Shape → X:{X_train.shape}  y:{y_train.shape}")
    print(f"  Train UP% → {(train_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Val   UP% → {(val_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Scalers saved → models/{symbol}/")

print(f"\n── prepared built for {len(prepared)} symbols ──")


── NVDA ────────────────────────────────────────────────────────────
  Rows  → train:5279  val:660  test:660
  Seqs  → train:5249  val:630  test:630
  Shape → X:(5249, 30, 58)  y:(5249, 1)
  Train UP% → 51.18%
  Val   UP% → 52.27%
  Scalers saved → models/NVDA/

── GOOGL ────────────────────────────────────────────────────────────
  Rows  → train:4160  val:520  test:521
  Seqs  → train:4130  val:490  test:491
  Shape → X:(4130, 30, 58)  y:(4130, 1)
  Train UP% → 52.26%
  Val   UP% → 51.73%
  Scalers saved → models/GOOGL/

── AAPL ────────────────────────────────────────────────────────────
  Rows  → train:5401  val:675  test:676
  Seqs  → train:5371  val:645  test:646
  Shape → X:(5371, 30, 58)  y:(5371, 1)
  Train UP% → 52.16%
  Val   UP% → 51.41%
  Scalers saved → models/AAPL/

── GOOG ────────────────────────────────────────────────────────────
  Rows  → train:2260  val:282  test:283
  Seqs  → train:2230  val:252  test:253
  Shape → X:(2230, 30, 58)  y:(2230, 1)
  Train UP% → 53.58

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class NASDAQDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = torch.tensor(weights, dtype=torch.float32) \
                 if weights is not None \
                 else torch.ones(len(X), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]


BATCH_SIZE = 32
loaders    = {}  

for symbol, data in prepared.items():
    train_loader = DataLoader(NASDAQDataset(data["X_train"], data["y_train"]), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(NASDAQDataset(data["X_val"],   data["y_val"]),   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(NASDAQDataset(data["X_test"],  data["y_test"]),  batch_size=BATCH_SIZE, shuffle=False)

    loaders[symbol] = {
        "train": train_loader,
        "val":   val_loader,
        "test":  test_loader,
    }

    print(f"\n{symbol} — Train: {len(train_loader)} batches | Val: {len(val_loader)} batches | Test: {len(test_loader)} batches")

    # ── Sanity check ──────────────────────────────────────────────────────────
    sample_X, sample_y, sample_w = next(iter(train_loader))   # ← unpack 3
    print(f"  Sample X: {sample_X.shape} → (batch, seq_len, features)")
    print(f"  Sample y: {sample_y.shape} → (batch, 4)")
    print(f"  Sample w: {sample_w.shape} → (batch,)  weights all 1.0 since no weights passed")


NVDA — Train: 165 batches | Val: 20 batches | Test: 20 batches
  Sample X: torch.Size([32, 30, 58]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed

GOOGL — Train: 130 batches | Val: 16 batches | Test: 16 batches
  Sample X: torch.Size([32, 30, 58]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed

AAPL — Train: 168 batches | Val: 21 batches | Test: 21 batches
  Sample X: torch.Size([32, 30, 58]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed

GOOG — Train: 70 batches | Val: 8 batches | Test: 8 batches
  Sample X: torch.Size([32, 30, 58]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 si

In [6]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

best_params_all = {}


# ── BiLSTM + Attention — single output (price_pct_change only) ────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm         = nn.LSTM(input_size, hidden_size, num_layers,
                                    batch_first=True, bidirectional=True,
                                    dropout=dropout if num_layers > 1 else 0)
        self.attention    = nn.Linear(hidden_size * 2, 1)
        self.pos_bias     = nn.Parameter(torch.zeros(1))
        self.ln           = nn.LayerNorm(hidden_size * 2)
        self.dropout      = nn.Dropout(dropout)
        self.fc           = nn.Linear(hidden_size * 2, 1)
        self.skip_fc      = nn.Linear(hidden_size * 2, 1)
        self.output_blend = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        seq_len     = lstm_out.size(1)

        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)

        out_attn = self.fc(self.dropout(self.ln(context)))
        out_skip = self.skip_fc(lstm_out[:, -1, :])

        alpha = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip   # (B, 1)


# ── Loss ──────────────────────────────────────────────────────────────────────
class PricePredictionLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=0.5, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma

    def forward(self, preds, targets, weights=None):
        p = preds[:, 0]
        t = targets[:, 0]

        huber     = nn.HuberLoss(delta=1.0, reduction="none")(p, t)
        sign_loss = torch.clamp(-p * t, min=0) * t.abs()

        pred_std       = p.std() + 1e-8
        tgt_std        = t.std() + 1e-8
        std_ratio      = pred_std / tgt_std
        std_match_loss = (pred_std - tgt_std).pow(2)
        collapse_loss  = torch.clamp(1.0 - std_ratio, min=0) ** 2

        pred_up_frac = torch.sigmoid(p * 2).mean()
        bias_penalty = (pred_up_frac - 0.5).pow(2) * 80.0

        per_sample = huber + self.alpha * sign_loss + 8.0 * std_match_loss
        if weights is not None:
            per_sample = per_sample * weights

        return (
            per_sample.mean()
            + self.beta  * (1.0 / std_ratio.clamp(min=0.05)).clamp(max=10.0)
            + self.gamma * collapse_loss * 5.0
            + bias_penalty
        )


# ── Helpers ───────────────────────────────────────────────────────────────────
def create_sequences(X, y, seq_len, move_threshold=0.3):
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        move = abs(y[i + seq_len, 0])
        ws.append(1.0 + 2.0 * float(move > move_threshold))
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


def make_balanced_sampler(y, oversample_factor=2.0):
    labels      = (y[:, 0] > 0).astype(int)
    moves       = np.abs(y[:, 0])
    median_move = np.median(moves)
    bucket      = labels * 2 + (moves > median_move).astype(int)
    counts      = np.maximum(np.bincount(bucket, minlength=4).astype(float), 1)
    weight_map  = 1.0 / counts
    weight_map[1] *= oversample_factor
    weight_map[3] *= oversample_factor
    sample_weights = torch.tensor(weight_map[bucket], dtype=torch.float32)
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


def safe_corr(a, b):
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else float(r)


# ── Clean NaN values before Optuna ────────────────────────────────────────────
print("\nCleaning NaN values from all symbols...")
for symbol, data in prepared.items():
    for split in ("train", "val"):
        X_key, y_key = f"{split}_X_scaled", f"{split}_y_scaled"
        X, y         = data[X_key], data[y_key]
        mask         = ~(np.isnan(X).any(axis=1) | np.isnan(y).any(axis=1))
        before       = len(X)
        data[X_key]  = X[mask]
        data[y_key]  = y[mask]
        print(f"  {symbol} {split}: {before} → {len(data[X_key])} rows")


# ── Optuna search ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*55}")
    print(f"  Optuna search — {symbol}")
    print(f"{'═'*55}")

    train_X    = data["train_X_scaled"]
    train_y    = data["train_y_scaled"]
    val_X      = data["val_X_scaled"]
    val_y      = data["val_y_scaled"]
    input_size = train_X.shape[1]

    def objective(trial):
        hidden_size    = trial.suggest_categorical("hidden_size", [64, 128])
        num_layers     = trial.suggest_int("num_layers", 2, 3)
        dropout        = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
        lr             = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        batch_size     = trial.suggest_categorical("batch_size", [32, 64])
        seq_len        = trial.suggest_categorical("seq_len", [10, 20, 30, 40])
        alpha          = trial.suggest_float("alpha", 3.0, 12.0, step=0.5)
        beta           = trial.suggest_float("beta", 0.3, 1.0, step=0.1)
        move_threshold = trial.suggest_float("move_threshold", 0.2, 0.5, step=0.1)
        gamma          = trial.suggest_float("gamma", 0.5, 3.0, step=0.5)

        # sequences built once — shared across seeds (deterministic)
        X_tr, y_tr, w_tr = create_sequences(train_X, train_y, seq_len, move_threshold)
        X_vl, y_vl, _    = create_sequences(val_X,   val_y,   seq_len, move_threshold)

        SEEDS       = [42, 123, 456]
        seed_scores = []

        for seed_idx, seed in enumerate(SEEDS):
            torch.manual_seed(seed)
            np.random.seed(seed)

            train_ld = DataLoader(
                NASDAQDataset(X_tr, y_tr, w_tr),
                batch_size=batch_size,
                sampler=make_balanced_sampler(y_tr),
                drop_last=True,
            )
            val_ld = DataLoader(
                NASDAQDataset(X_vl, y_vl),
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )

            model = StockPctChangeBiLSTMAttention(
                input_size=input_size, hidden_size=hidden_size,
                num_layers=num_layers, dropout=dropout,
            ).to(device)

            criterion = PricePredictionLoss(alpha=alpha, beta=beta, gamma=gamma)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

            EPOCHS         = 50
            PATIENCE       = 10
            best_val_score = -1.0
            patience_ctr   = 0
            collapsed      = False

            for epoch in range(1, EPOCHS + 1):
                model.train()
                for X_batch, y_batch, w_batch in train_ld:
                    X_batch, y_batch, w_batch = X_batch.to(device), y_batch.to(device), w_batch.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(X_batch), y_batch, w_batch)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                model.eval()
                all_preds, all_actuals = [], []
                with torch.no_grad():
                    for X_batch, y_batch, _ in val_ld:
                        all_preds.append(model(X_batch.to(device)).cpu().numpy())
                        all_actuals.append(y_batch.numpy())

                if not all_preds:
                    collapsed = True
                    break

                val_preds   = np.vstack(all_preds)
                val_actuals = np.vstack(all_actuals)

                pred_std_ratio = np.std(val_preds[:, 0]) / max(np.std(val_actuals[:, 0]), 1e-6)
                pred_up_pct    = (val_preds[:, 0] > 0).mean()
                actual_up_pct  = (val_actuals[:, 0] > 0).mean()

                if epoch >= 5:
                    if pred_up_pct < 0.15 or pred_up_pct > 0.85:
                        collapsed = True
                        break
                    if pred_std_ratio < 0.30:
                        collapsed = True
                        break

                price_dir   = balanced_accuracy_score(
                    np.sign(val_actuals[:, 0]).astype(int),
                    np.sign(val_preds[:, 0]).astype(int),
                )
                price_corr      = safe_corr(val_actuals[:, 0], val_preds[:, 0])
                bias_gap        = abs(pred_up_pct - actual_up_pct)
                balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0

                val_score = (
                    0.70 * price_dir +
                    0.30 * max(price_corr, 0.0) -
                    abs(1.0 - pred_std_ratio) * 0.3 -
                    bias_gap * 3.0 -
                    balance_penalty
                )

                if val_score > best_val_score:
                    best_val_score = val_score
                    patience_ctr   = 0
                else:
                    patience_ctr += 1
                    if patience_ctr >= PATIENCE:
                        break

            seed_scores.append(-1.0 if collapsed else best_val_score)

            # prune after each seed based on mean so far
            trial.report(float(np.mean(seed_scores)), seed_idx)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return float(np.mean(seed_scores))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
        study_name=f"bilstm_price_{symbol}",
    )
    study.optimize(objective, n_trials=50, timeout=3600)

    best_params_all[symbol] = study.best_params

    print(f"\n  Best val score : {study.best_value:.4f}")
    print(f"  Best trial     : {study.best_trial.number}")
    print(f"  Best params:")
    for k, v in study.best_params.items():
        print(f"    {k:15s} : {v}")

    trials_df = study.trials_dataframe().sort_values("value", ascending=False).head()
    cols = [
        "number", "value",
        "params_hidden_size", "params_num_layers", "params_dropout",
        "params_lr", "params_batch_size", "params_seq_len",
        "params_alpha", "params_beta", "params_move_threshold",
    ]
    print(f"\n  Top 5 Trials:")
    print(trials_df[[c for c in cols if c in trials_df.columns]])

print(f"\n── Optuna done for {len(best_params_all)} symbols ──")

Using device: cuda

[I 2026-06-05 00:51:44,692] A new study created in memory with name: bilstm_price_NVDA




Cleaning NaN values from all symbols...
  NVDA train: 5279 → 5279 rows
  NVDA val: 660 → 660 rows
  GOOGL train: 4160 → 4160 rows
  GOOGL val: 520 → 520 rows
  AAPL train: 5401 → 5401 rows
  AAPL val: 675 → 675 rows
  GOOG train: 2260 → 2260 rows
  GOOG val: 282 → 282 rows
  MSFT train: 5388 → 5388 rows
  MSFT val: 674 → 674 rows
  AVGO train: 3182 → 3182 rows
  AVGO val: 398 → 398 rows
  TSLA train: 2996 → 2996 rows
  TSLA val: 375 → 375 rows
  META train: 2632 → 2632 rows
  META val: 329 → 329 rows

═══════════════════════════════════════════════════════
  Optuna search — NVDA
═══════════════════════════════════════════════════════


[I 2026-06-05 00:52:17,068] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-05 00:52:45,855] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 0 with value: -1.0.
[I 2026-06-05 00:53:16,369] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-05 00:53:46,889] Trial 3 finished with value: -1.0 and paramet


  Best val score : -1.0000
  Best trial     : 0
  Best params:
    hidden_size     : 128
    num_layers      : 3
    dropout         : 0.30000000000000004
    lr              : 0.00018410729205738696
    batch_size      : 32
    seq_len         : 10
    alpha           : 12.0
    beta            : 0.9000000000000001
    move_threshold  : 0.2
    gamma           : 1.0

  Top 5 Trials:
    number  value  params_hidden_size  params_num_layers  params_dropout  \
0        0   -1.0                 128                  3             0.3   
37      37   -1.0                 128                  3             0.3   
27      27   -1.0                 128                  3             0.4   
28      28   -1.0                  64                  2             0.4   
29      29   -1.0                  64                  3             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
0    0.000184                 32              10          12.0          0.9 

...


  Best val score : -1.0000
  Best trial     : 0
  Best params:
    hidden_size     : 128
    num_layers      : 3
    dropout         : 0.30000000000000004
    lr              : 0.00018410729205738696
    batch_size      : 32
    seq_len         : 10
    alpha           : 12.0
    beta            : 0.9000000000000001
    move_threshold  : 0.2
    gamma           : 1.0

  Top 5 Trials:
    number  value  params_hidden_size  params_num_layers  params_dropout  \
0        0   -1.0                 128                  3             0.3   
37      37   -1.0                 128                  3             0.3   
27      27   -1.0                 128                  3             0.4   
28      28   -1.0                  64                  2             0.4   
29      29   -1.0                  64                  3             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
0    0.000184                 32              10          12.0          0.9 

[I 2026-06-05 02:56:18,667] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-05 02:56:37,162] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 0 with value: -1.0.
[I 2026-06-05 02:56:54,267] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-05 02:57:11,945] Trial 3 finished with value: -1.0 and paramet


  Best val score : -0.5409
  Best trial     : 41
  Best params:
    hidden_size     : 128
    num_layers      : 2
    dropout         : 0.4
    lr              : 0.00010167634056406136
    batch_size      : 32
    seq_len         : 10
    alpha           : 3.5
    beta            : 0.3
    move_threshold  : 0.5
    gamma           : 2.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
41      41 -0.540911                 128                  2             0.4   
43      43 -0.549731                 128                  2             0.4   
38      38 -0.550541                 128                  2             0.4   
45      45 -0.554421                 128                  2             0.4   
36      36 -0.556269                 128                  2             0.4   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
41   0.000102                 32              10           3.5          0.3   
43   0.000

[I 2026-06-05 03:13:36,994] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-05 03:13:51,832] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 0 with value: -1.0.
[I 2026-06-05 03:14:05,891] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-05 03:14:20,897] Trial 3 finished with value: -1.0 and paramet


  Best val score : -1.0000
  Best trial     : 0
  Best params:
    hidden_size     : 128
    num_layers      : 3
    dropout         : 0.30000000000000004
    lr              : 0.00018410729205738696
    batch_size      : 32
    seq_len         : 10
    alpha           : 12.0
    beta            : 0.9000000000000001
    move_threshold  : 0.2
    gamma           : 1.0

  Top 5 Trials:
    number  value  params_hidden_size  params_num_layers  params_dropout  \
0        0   -1.0                 128                  3             0.3   
37      37   -1.0                 128                  3             0.3   
27      27   -1.0                 128                  3             0.4   
28      28   -1.0                  64                  2             0.4   
29      29   -1.0                  64                  3             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
0    0.000184                 32              10          12.0          0.9 

In [7]:
# Training
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import balanced_accuracy_score
import numpy as np   
import os
import time

trained_models = {}


def compute_val_metrics(model, loader, device):
    """Single forward pass — price only. Score formula identical to Optuna objective."""
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in loader:
            all_preds.append(model(X_batch.to(device)).cpu().numpy())
            all_actuals.append(y_batch.numpy())

    preds   = np.vstack(all_preds)   # (N, 1)
    actuals = np.vstack(all_actuals)  # (N, 1)

    def safe_corr(a, b):
        if np.std(a) < 1e-8 or np.std(b) < 1e-8:
            return 0.0
        r = np.corrcoef(a, b)[0, 1]
        return 0.0 if np.isnan(r) else float(r)

    price_dir  = balanced_accuracy_score(
        np.sign(actuals[:, 0]).astype(int),
        np.sign(preds[:, 0]).astype(int),
    )
    price_corr  = safe_corr(actuals[:, 0], preds[:, 0])
    pred_up_pct = (preds[:, 0] > 0).mean()
    std_ratio   = np.std(preds[:, 0]) / max(np.std(actuals[:, 0]), 1e-6)

    bias_gap        = abs(pred_up_pct - 0.5)
    balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0

    joint_score = (
        0.70 * price_dir            +
        0.30 * max(price_corr, 0.0) -
        abs(1.0 - std_ratio) * 0.3  -
        bias_gap * 3.0              -
        balance_penalty
    )

    return dict(
        joint_score=joint_score,
        price_dir=price_dir,
        price_corr=price_corr,
        pred_up_pct=pred_up_pct,
        std_ratio=std_ratio,
    )


# ── Training loop ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*75}")
    print(f"  Training — {symbol}")
    print(f"{'═'*75}")

    best = best_params_all[symbol]
    print(f"  Best params: {best}")

    move_threshold  = best.get("move_threshold", 0.3)
    BASE_BCE_WEIGHT = best.get("bce_weight", 10.0)

    X_train_f, y_train_f, w_train_f = create_sequences(
        data["train_X_scaled"], data["train_y_scaled"], best["seq_len"], move_threshold)
    X_val_f,   y_val_f,   _         = create_sequences(
        data["val_X_scaled"],   data["val_y_scaled"],   best["seq_len"], move_threshold)
    X_test_f,  y_test_f,  _         = create_sequences(
        data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"], move_threshold)

    train_loader = DataLoader(
        NASDAQDataset(X_train_f, y_train_f, w_train_f),
        batch_size=best["batch_size"],
        sampler=make_balanced_sampler(y_train_f),
        drop_last=True,
    )
    val_loader  = DataLoader(NASDAQDataset(X_val_f,  y_val_f),  batch_size=best["batch_size"], shuffle=False)
    test_loader = DataLoader(NASDAQDataset(X_test_f, y_test_f), batch_size=best["batch_size"], shuffle=False)

    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = best["hidden_size"],
        num_layers  = best["num_layers"],
        dropout     = best["dropout"],
    ).to(device)

    WARMUP_EPOCHS = 10
    BASE_ALPHA    = best["alpha"]
    BASE_BETA     = best["beta"]
    BASE_GAMMA    = best.get("gamma", 1.0)

    optimizer = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

    EPOCHS           = 100
    PATIENCE         = 15
    best_joint_score = -np.inf
    patience_ctr     = 0
    collapse_ctr     = 0
    history = {
        "train_loss":  [],
        "joint_score": [],
        "price_dir":   [],
        "price_corr":  [],
        "pred_up_pct": [],
        "std_ratio":   [],
    }

    os.makedirs(f"models/{symbol}", exist_ok=True)
    save_path = f"models/{symbol}/{symbol}_best_bilstm.pt"

    print(f"\n{'Epoch':>6} | {'TrLoss':>8} | {'Joint':>7} | "
        f"{'PDir':>6} | {'Pr':>6} | {'UP%':>5} | {'StdR':>5} | {'s':>5}")
    print("─" * 65)

    for epoch in range(1, EPOCHS + 1):
        start        = time.time()
        warmup_scale = 0.3 + 0.7 * min(1.0, epoch / WARMUP_EPOCHS)
        criterion    = PricePredictionLoss(
            alpha      = BASE_ALPHA * warmup_scale,
            beta       = BASE_BETA,
            gamma      = BASE_GAMMA,
        )

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for X_batch, y_batch, w_batch in train_loader:
            X_batch, y_batch, w_batch = X_batch.to(device), y_batch.to(device), w_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        # ── Val metrics — single forward pass ─────────────────────────────────
        m = compute_val_metrics(model, val_loader, device)

        elapsed = time.time() - start

        history["train_loss"].append(train_loss)
        history["joint_score"].append(m["joint_score"])
        history["price_dir"].append(m["price_dir"])
        history["price_corr"].append(m["price_corr"])
        history["pred_up_pct"].append(m["pred_up_pct"])
        history["std_ratio"].append(m["std_ratio"])

        print(f"{epoch:>6} | {train_loss:>8.5f} | {m['joint_score']:>7.4f} | "
            f"{m['price_dir']:>6.3f} | {m['price_corr']:>6.3f} | "
            f"{m['pred_up_pct']:>4.0%} | {m['std_ratio']:>5.2f} | {elapsed:>4.1f}s")

        # ── Collapse detection ─────────────────────────────────────────────────
        if epoch > WARMUP_EPOCHS:
            if m["pred_up_pct"] > 0.85 or m["pred_up_pct"] < 0.15 or m["std_ratio"] < 0.20:
                collapse_ctr += 1
                if collapse_ctr >= 3:
                    print(f"\n  Collapse at epoch {epoch} "
                        f"(up%={m['pred_up_pct']:.0%}, std_r={m['std_ratio']:.2f}) — resetting weights")
                    model.apply(lambda mod: mod.reset_parameters() if hasattr(mod, 'reset_parameters') else None)
                    for pg in optimizer.param_groups:
                        pg['lr'] = best["lr"]
                    collapse_ctr = 0
            else:
                collapse_ctr = 0

        scheduler.step(m["joint_score"])

        if m["joint_score"] > best_joint_score:
            best_joint_score = m["joint_score"]
            patience_ctr     = 0
            torch.save(model.state_dict(), save_path)
            print(f"         ✓ Saved  "
                f"(p_dir={m['price_dir']:.3f}, p_corr={m['price_corr']:.3f}, "
                f"up%={m['pred_up_pct']:.0%}, std_r={m['std_ratio']:.2f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch}.")
                break

    trained_models[symbol] = {
        "best_joint_score": best_joint_score,
        "history":          history,
    }
    print(f"\n  Best joint score: {best_joint_score:.4f}")

print(f"\n── Training done for {len(trained_models)} symbols ──")
for s, r in trained_models.items():
    print(f"  {s}: best joint score = {r['best_joint_score']:.4f}")


═══════════════════════════════════════════════════════════════════════════
  Training — NVDA
═══════════════════════════════════════════════════════════════════════════
  Best params: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}

 Epoch |   TrLoss |   Joint |   PDir |     Pr |   UP% |  StdR |     s
─────────────────────────────────────────────────────────────────
     1 | 11.88048 | -3.3395 |  0.500 | -0.025 |   0% |  0.37 |  2.5s
         ✓ Saved  (p_dir=0.500, p_corr=-0.025, up%=0%, std_r=0.37)
     2 | 10.57430 | -3.3012 |  0.500 | -0.019 |   0% |  0.50 |  2.4s
         ✓ Saved  (p_dir=0.500, p_corr=-0.019, up%=0%, std_r=0.50)
     3 | 10.51034 | -2.3777 |  0.507 |  0.005 |   8% |  1.21 |  2.3s
         ✓ Saved  (p_dir=0.507, p_corr=0.005, up%=8%, std_r=1.21)
     4 | 11.25538 | -3.2993 |  0.500 | -0.010 |   0% |  0

In [8]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────
import pickle

print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)


def evaluate(actuals, preds, label):
    mae         = np.mean(np.abs(actuals - preds))
    rmse        = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc     = np.mean(np.sign(actuals) == np.sign(preds)) * 100
    bal_dir_acc = balanced_accuracy_score(
        np.sign(actuals).astype(int), np.sign(preds).astype(int)
    ) * 100
    corr        = np.corrcoef(actuals, preds)[0, 1] if np.std(preds) > 1e-8 else 0.0
    pred_std    = np.std(preds)
    actual_std  = np.std(actuals)
    within_1pct = np.mean(np.abs(actuals - preds) < 1.0) * 100
    within_2pct = np.mean(np.abs(actuals - preds) < 2.0) * 100

    print(f"\n  [{label}]")
    print(f"  MAE             : {mae:.4f}%")
    print(f"  RMSE            : {rmse:.4f}%")
    print(f"  Pearson r       : {corr:.4f}")
    print(f"  Pred std        : {pred_std:.4f}  Actual std: {actual_std:.4f}")
    print(f"  Within ±1%      : {within_1pct:.1f}%")
    print(f"  Within ±2%      : {within_2pct:.1f}%")
    print(f"  Directional Acc : {dir_acc:.2f}%")
    print(f"  Balanced Dir Acc: {bal_dir_acc:.2f}%")


for symbol in symbols:
    if symbol not in prepared:
        print(f"\n  Skipping {symbol} — not in prepared dict.")
        continue
    if symbol not in trained_models:
        print(f"\n  Skipping {symbol} — no trained model found.")
        continue

    print(f"\n{'═'*55}")
    print(f"  Evaluating — {symbol}")
    print(f"{'═'*55}")

    best           = best_params_all[symbol]
    data           = prepared[symbol]
    move_threshold = best.get("move_threshold", 0.3)

    X_test_f, y_test_f, _ = create_sequences(
        data["test_X_scaled"], data["test_y_scaled"], best["seq_len"], move_threshold
    )
    test_loader = DataLoader(
        NASDAQDataset(X_test_f, y_test_f),
        batch_size=best["batch_size"],
        shuffle=False,
    )

    checkpoint = torch.load(
        f"models/{symbol}/{symbol}_best_bilstm.pt",
        weights_only=True,
    )

    inferred_hidden_size = checkpoint["lstm.weight_ih_l0"].shape[0] // 4
    inferred_num_layers  = sum(
        1 for k in checkpoint
        if k.startswith("lstm.weight_ih_l") and "_reverse" not in k
    )

    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = inferred_hidden_size,
        num_layers  = inferred_num_layers,
        dropout     = best["dropout"],
    ).to(device)
    model.load_state_dict(checkpoint)
    model.eval()

    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "rb") as f:
        target_scaler = pickle.load(f)

    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:
            all_preds.append(model(X_batch.to(device)).cpu().numpy())
            all_actuals.append(y_batch.numpy())

    all_preds   = target_scaler.inverse_transform(np.vstack(all_preds))
    all_actuals = target_scaler.inverse_transform(np.vstack(all_actuals))

    price_preds_pct   = all_preds[:, 0]
    price_actuals_pct = all_actuals[:, 0]

    evaluate(price_actuals_pct, price_preds_pct, "Price % Change")

    results_df = pd.DataFrame({
        "actual_price_pct"   : price_actuals_pct,
        "predicted_price_pct": price_preds_pct,
    })
    out_path = f"models/{symbol}/{symbol}_test_predictions.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\n  Predictions saved: {out_path}")

print(f"\n── Evaluation done for {len(symbols)} symbols ──")


EVALUATION ON TEST SET

═══════════════════════════════════════════════════════
  Evaluating — NVDA
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 2.1387%
  RMSE            : 2.7929%
  Pearson r       : -0.0195
  Pred std        : 0.3517  Actual std: 2.7424
  Within ±1%      : 31.2%
  Within ±2%      : 56.2%
  Directional Acc : 48.00%
  Balanced Dir Acc: 48.41%

  Predictions saved: models/NVDA/NVDA_test_predictions.csv

═══════════════════════════════════════════════════════
  Evaluating — GOOGL
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 2.4123%
  RMSE            : 3.0452%
  Pearson r       : -0.0190
  Pred std        : 1.9301  Actual std: 1.6791
  Within ±1%      : 30.3%
  Within ±2%      : 51.7%
  Directional Acc : 48.73%
  Balanced Dir Acc: 48.82%

  Predictions saved: models/GOOGL/GOOGL_test_predictions.csv

═══════════════════════════════════════════════════════
  Evaluating — A

In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score
                    
for symbol in symbols:
    csv_path = f"models/{symbol}/{symbol}_test_predictions.csv"

    try:
        results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"\n  Skipping {symbol} — {csv_path} not found.")
        continue

    price_actual    = results["actual_price_pct"]
    price_predicted = results["predicted_price_pct"]

    correct   = (np.sign(price_actual) == np.sign(price_predicted))
    up_mask   = price_actual > 0
    down_mask = price_actual < 0
    bal_acc   = balanced_accuracy_score(
        np.sign(price_actual).astype(int),
        np.sign(price_predicted).astype(int),
    )

    print(f"\n{'═'*60}")
    print(f"  {symbol} — Diagnostics")
    print(f"{'═'*60}")
    print(f"  Actual    mean : {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
    print(f"  Predicted mean : {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")
    print(f"\n  Overall Dir Acc : {correct.mean()*100:.2f}%")
    print(f"  Balanced Dir Acc: {bal_acc*100:.2f}%")
    print(f"  When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
    print(f"  When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")
    print(f"\n  % predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
    print(f"  % actual UP     : {(price_actual > 0).mean()*100:.2f}%")

    print(f"\n  Dir Acc by Move Size (NASDAQ-adjusted):")
    bins   = [0, 0.5, 1.0, 2.0, 5.0, 10.0, 100.0]
    labels = ["<0.5%", "0.5–1%", "1–2%", "2–5%", "5–10%", ">10%"]
    price_actual_abs = price_actual.abs()

    for i, label in enumerate(labels):
        mask = (price_actual_abs >= bins[i]) & (price_actual_abs < bins[i + 1])
        n    = mask.sum()
        if n == 0:
            continue
        acc  = correct[mask].mean() * 100
        flag = "  ⚠ below chance" if (acc < 50.0 and bins[i] >= 1.0) else ""
        print(f"    {label:>8} moves : {acc:.2f}%  ({n} samples){flag}")

print(f"\n── Diagnostics done for {len(symbols)} symbols ──")



════════════════════════════════════════════════════════════
  NVDA — Diagnostics
════════════════════════════════════════════════════════════
  Actual    mean : 0.2457   std: 2.7445
  Predicted mean : -0.0978   std: 0.3520

  Overall Dir Acc : 48.00%
  Balanced Dir Acc: 48.41%
  When actual UP  : 44.26%  (357 samples)
  When actual DOWN: 52.56%  (293 samples)

  % predicted UP  : 45.69%
  % actual UP     : 54.92%

  Dir Acc by Move Size (NASDAQ-adjusted):
       <0.5% moves : 52.83%  (106 samples)
      0.5–1% moves : 49.09%  (110 samples)
        1–2% moves : 47.24%  (163 samples)  ⚠ below chance
        2–5% moves : 48.46%  (227 samples)  ⚠ below chance
       5–10% moves : 34.88%  (43 samples)  ⚠ below chance
        >10% moves : 0.00%  (1 samples)  ⚠ below chance

════════════════════════════════════════════════════════════
  GOOGL — Diagnostics
════════════════════════════════════════════════════════════
  Actual    mean : 0.1243   std: 1.6807
  Predicted mean : -1.4898   std: 1